# L0b step 2 · The grader and L0b together: two rules fixed before scoring (your PC, no GPU)

Built 22 Nov 2026 at Abrar's go. It reads the answers L0b's run saved
([L0b-results/L0b/l0b_answers.csv](L0b-results/L0b/l0b_answers.csv)) and runs no model.

**Why:** on Part B's 300 validation buildings (100 per class), the grader is right on 212 and L0b (our 7B with its
language and upper vision layers trained on building crops) on 176. Their mistakes are opposite. The grader gets 92 of
100 intact buildings but only 66 of 100 destroyed ones; L0b gets 86 destroyed but only 40 intact, since it calls many
intact buildings damaged (44) or destroyed (16). At least one of the two is right on 259 buildings. A rule that uses
L0b only where it's strong might keep the grader's intact calls and add L0b's destroyed ones.

**The two rules, fixed before any combined score was computed:**
- **Rule A:** destroyed if L0b says destroyed; otherwise the grader's answer.
- **Rule B:** destroyed if L0b says destroyed *and* the grader says damaged; otherwise the grader's answer.

The grader's 16 "not a building" answers stay as they are (wrong) under both rules, as in its 212.

**How they're scored:**
- buildings right of 300, per class, against the grader with McNemar's exact test (only the buildings where they
  differ count);
- **weighted by real class shares.** The 300 have 100 of each class, but the validation set's 25,594 true buildings
  are 76.7% intact, 17.1% damaged and 6.3% destroyed (R16c step 7's `true_grades.csv`). A wrong "destroyed" on an
  intact building costs far more in a real photo than on this sample, so accuracy and destroyed precision are also
  given at those shares;
- destroyed F1 (precision and recall together) at real shares, since destroyed buildings are what the project and the
  reports count first.

**Worth a confirmation run** on new buildings (L0b on buildings it hasn't seen, about 1 GPU hour) only if a rule:
1. has more buildings right than the grader on the 300; or
2. raises destroyed F1 at real shares by at least 0.05 while lowering accuracy at real shares by at most 1 point.

Scoring both rules on the same 300 buildings where the pattern was spotted flatters them, which is why a pass only
earns a confirmation run, not a change to the chain.

**Expectation, fixed before scoring** (from L0b's own confusion table, already published: it says destroyed on 16
intact, 33 damaged and 86 destroyed buildings):
- **Rule A:** destroyed about 88–91 right, intact down to about 76–80, damaged down to about 35–45; about 205–215 in
  all. At real shares, accuracy drops by about 10 points (from about 84%), so it fails criterion 2.
- **Rule B:** destroyed up a little (about 75–85), intact unchanged (92), damaged down (about 40–50); about 208–218 in
  all; accuracy at real shares down by 1–3 points.
- Neither is predicted to earn the confirmation run.

In [1]:
# ── Cell 1: the saved answers, one row per building ──
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest

CT = Path(r"D:\CSE499AB_project\Essential_task_delegation\Champion_Task")
OUT = CT / "L0b-results" / "step2"
OUT.mkdir(parents=True, exist_ok=True)
CLASSES = ["Intact", "Damaged", "Destroyed"]

ans = pd.read_csv(CT / "L0b-results" / "L0b" / "l0b_answers.csv")
wide = ans.pivot(index="item", columns="run", values="vlm_class")
wide["true"] = ans[ans.run == "grader"].set_index("item")["true_class"]
B = wide[["true", "grader", "l0b_vision_lora"]].rename(columns={"l0b_vision_lora": "l0b"})
right = {m: int((B[m] == B["true"]).sum()) for m in ("grader", "l0b")}
assert right == {"grader": 212, "l0b": 176}, right            # the published numbers
assert B["true"].value_counts().to_dict() == {c: 100 for c in CLASSES}
print(f"✓ {len(B)} buildings; grader {right['grader']}, L0b {right['l0b']} right; "
      f"the grader gives no class on {int(B['grader'].isna().sum())}")

tg = pd.read_csv(CT / "R16c-results" / "step7" / "true_grades.csv")
tg = tg[(tg.variant == "V0") & (tg.seed == 42)]                # one row per true building
SHARE = (tg["true"].value_counts(normalize=True).sort_index().rename(dict(enumerate(CLASSES)))).to_dict()
print(f"✓ real class shares on {len(tg):,} validation buildings:", {k: round(v, 4) for k, v in SHARE.items()})

✓ 300 buildings; grader 212, L0b 176 right; the grader gives no class on 16


✓ real class shares on 25,594 validation buildings: {'Intact': 0.7667, 'Damaged': 0.1707, 'Destroyed': 0.0625}


In [2]:
# ── Cell 2: the two rules, scored ──
B["rule_A"] = np.where(B["l0b"] == "Destroyed", "Destroyed", B["grader"])
B["rule_B"] = np.where((B["l0b"] == "Destroyed") & (B["grader"] == "Damaged"), "Destroyed", B["grader"])
B.loc[B["grader"].isna() & (B["rule_A"] != "Destroyed"), "rule_A"] = np.nan   # "not a building" stays as it was
B.loc[B["grader"].isna(), "rule_B"] = np.nan


def scores(m):
    ok = B[m] == B["true"]
    per = {c: int(ok[B["true"] == c].sum()) for c in CLASSES}
    rate = {c: float((B.loc[B["true"] == c, m] == "Destroyed").mean()) for c in CLASSES}   # says destroyed, by true class
    recall = rate["Destroyed"]
    called = sum(SHARE[c] * rate[c] for c in CLASSES)
    precision = SHARE["Destroyed"] * recall / called if called else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"method": m, "right": int(ok.sum()), **{f"{c.lower()}_right": per[c] for c in CLASSES},
            "accuracy_real_shares": round(sum(SHARE[c] * per[c] / 100 for c in CLASSES), 4),
            "destroyed_recall": round(recall, 3), "destroyed_precision_real_shares": round(precision, 3),
            "destroyed_f1_real_shares": round(f1, 3)}


def mcnemar(m):
    a, b = B[m] == B["true"], B["grader"] == B["true"]
    only_m, only_g = int((a & ~b).sum()), int((~a & b).sum())
    return {"only_rule_right": only_m, "only_grader_right": only_g,
            "mcnemar_p": round(binomtest(only_m, only_m + only_g, 0.5).pvalue, 4) if only_m + only_g else 1.0}


TABLE = pd.DataFrame([scores(m) for m in ("grader", "l0b", "rule_A", "rule_B")])
for m in ("rule_A", "rule_B"):
    for k, v in mcnemar(m).items():
        TABLE.loc[TABLE.method == m, k] = v
print(TABLE.to_string(index=False))

method  right  intact_right  damaged_right  destroyed_right  accuracy_real_shares  destroyed_recall  destroyed_precision_real_shares  destroyed_f1_real_shares  only_rule_right  only_grader_right  mcnemar_p
grader    212            92             54               66                0.8389              0.66                            0.567                     0.610              NaN                NaN        NaN
   l0b    176            40             50               86                0.4458              0.86                            0.231                     0.364              NaN                NaN        NaN
rule_A    207            76             40               91                0.7079              0.91                            0.217                     0.350             25.0               30.0     0.5901
rule_B    211            92             40               79                0.8231              0.79                            0.471                     0.590             13.0 

In [3]:
# ── Cell 3: the criteria fixed before scoring, and the files ──
g = TABLE.set_index("method").loc["grader"]
lines = []
for m in ("rule_A", "rule_B"):
    r = TABLE.set_index("method").loc[m]
    c1 = r["right"] > g["right"]
    c2 = (r["destroyed_f1_real_shares"] - g["destroyed_f1_real_shares"] >= 0.05
          and g["accuracy_real_shares"] - r["accuracy_real_shares"] <= 0.01)
    verdict = "earns a confirmation run" if (c1 or c2) else "doesn't earn a confirmation run"
    lines.append(f"- **{m.replace('_', ' ').title()}: {verdict}.** {int(r['right'])} right of 300 (grader {int(g['right'])}; "
                 f"McNemar p = {r['mcnemar_p']}); at real shares, accuracy {r['accuracy_real_shares']:.1%} (grader "
                 f"{g['accuracy_real_shares']:.1%}) and destroyed F1 {r['destroyed_f1_real_shares']} (grader "
                 f"{g['destroyed_f1_real_shares']}).")
print("\n".join(lines))


def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


B.reset_index().to_csv(OUT / "per_building.csv", index=False)
TABLE.to_csv(OUT / "scores.csv", index=False)
(OUT / "README.md").write_text("\n".join([
    "# L0b step 2: the grader and L0b together", "",
    "Made by [l0b-step2-grader-combination.ipynb](../../l0b-step2-grader-combination.ipynb), on your PC, from L0b's saved "
    "answers. The rules, the scoring, the criteria and the expectation were fixed in its first cell before scoring.", "",
    *lines, "", "## Scores", "", md(TABLE), "",
    f"Real class shares used: {', '.join(f'{k} {v:.1%}' for k, v in SHARE.items())} (25,594 validation buildings).", "",
    "Files: `per_building.csv` (each building's true class and the four answers), `scores.csv`."]) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md, scores.csv and per_building.csv to {OUT}")

- **Rule A: doesn't earn a confirmation run.** 207 right of 300 (grader 212; McNemar p = 0.5901); at real shares, accuracy 70.8% (grader 83.9%) and destroyed F1 0.35 (grader 0.61).
- **Rule B: doesn't earn a confirmation run.** 211 right of 300 (grader 212; McNemar p = 1.0); at real shares, accuracy 82.3% (grader 83.9%) and destroyed F1 0.59 (grader 0.61).
✓ Wrote README.md, scores.csv and per_building.csv to D:\CSE499AB_project\Essential_task_delegation\Champion_Task\L0b-results\step2
